# Hand Gesture Recognition — Live Streaming Demo

This notebook runs a **real-time hand gesture recognition system** entirely inside Google Colab.

**Note:** Expect ~2-5 FPS due to the JS↔Python round-trip. This is a Colab limitation, not a MediaPipe issue.



## Step 1: Install Dependencies

We need to manually manage the protobuf version because Colab's default environment ships with a newer protobuf that conflicts with mediapipe==0.10.13.This must be done every session since Colab resets the environment on each new runtime.

**Note:** After running this cell, session needs to be restarted then we run
the cells again from the top.



In [ ]:
!pip uninstall -y protobuf
!pip install protobuf==4.25.3 --no-deps
!pip install mediapipe==0.10.30 opencv-python-headless numpy tensorflow==2.20.0


## Step 2: Clone the Repository


In [2]:
!git clone https://github.com/Igri04/Vision-Based-Hand-Gesture-Music-Generation.git

import os
os.chdir('Vision-Based-Hand-Gesture-Music-Generation')
print('Working directory:', os.getcwd())


Cloning into 'Vision-Based-Hand-Gesture-Music-Generation'...
remote: Enumerating objects: 48, done.
remote: Counting objects: 100% (48/48), done.
remote: Compressing objects: 100% (46/46), done.
remote: Total 48 (delta 10), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (48/48), 3.73 MiB | 4.39 MiB/s, done.
Resolving deltas: 100% (10/10), done.
Working directory: /content/Vision-Based-Hand-Gesture-Music-Generation


## Step 3: Imports

Imports for reading gesture label CSV files, deep copying image frames,
encoding/decoding frames between JS and Python, controlling sound cooldown timing, working with arrays, and models

In [3]:
import csv
import copy
import itertools
import base64
import time
from collections import Counter, deque

import cv2
import numpy as np
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision
from IPython.display import display, Javascript
from google.colab.output import eval_js

from model import KeyPointClassifier
from model import PointHistoryClassifier

print('Imports successful.')


Imports successful.


## Step 4: Helper Functions

These functions handle the visual processing of each frame — computing bounding boxes, normalizing landmark coordinates for the classifier, and drawing results back onto the image.



In [4]:
def calc_bounding_rect(image, landmarks):
    image_width, image_height = image.shape[1], image.shape[0]
    landmark_array = np.empty((0, 2), int)
    for _, landmark in enumerate(landmarks):
        landmark_x = min(int(landmark.x * image_width), image_width - 1)
        landmark_y = min(int(landmark.y * image_height), image_height - 1)
        landmark_array = np.append(landmark_array, [np.array((landmark_x, landmark_y))], axis=0)
    x, y, w, h = cv2.boundingRect(landmark_array)
    return [x, y, x + w, y + h]

def calc_landmark_list(image, landmarks):
    image_width, image_height = image.shape[1], image.shape[0]
    landmark_point = []
    for _, landmark in enumerate(landmarks):
        landmark_x = min(int(landmark.x * image_width), image_width - 1)
        landmark_y = min(int(landmark.y * image_height), image_height - 1)
        landmark_point.append([landmark_x, landmark_y])
    return landmark_point

def pre_process_landmark(landmark_list):
    temp = copy.deepcopy(landmark_list)
    base_x, base_y = temp[0][0], temp[0][1]
    for i in range(len(temp)):
        temp[i][0] -= base_x
        temp[i][1] -= base_y
    temp = list(itertools.chain.from_iterable(temp))
    max_val = max(map(abs, temp))
    return [n / max_val for n in temp]

def pre_process_point_history(image, point_history):
    iw, ih = image.shape[1], image.shape[0]
    temp = copy.deepcopy(point_history)
    if len(temp) == 0:
        return []
    base_x, base_y = temp[0][0], temp[0][1]
    for i in range(len(temp)):
        temp[i][0] = (temp[i][0] - base_x) / iw
        temp[i][1] = (temp[i][1] - base_y) / ih
    return list(itertools.chain.from_iterable(temp))

def draw_landmarks(image, lp):
    if len(lp) > 0:
        connections = [(2,3),(3,4),(5,6),(6,7),(7,8),(9,10),(10,11),(11,12),
                       (13,14),(14,15),(15,16),(17,18),(18,19),(19,20),
                       (0,1),(1,2),(2,5),(5,9),(9,13),(13,17),(17,0)]
        for s, e in connections:
            cv2.line(image, tuple(lp[s]), tuple(lp[e]), (0,0,0), 6)
            cv2.line(image, tuple(lp[s]), tuple(lp[e]), (255,255,255), 2)
        for i, p in enumerate(lp):
            r = 8 if i in [4,8,12,16,20] else 5
            cv2.circle(image, tuple(p), r, (255,255,255), -1)
            cv2.circle(image, tuple(p), r, (0,0,0), 1)
    return image

def draw_bounding_rect(image, brect):
    cv2.rectangle(image, (brect[0], brect[1]), (brect[2], brect[3]), (0,0,0), 1)
    return image

def draw_info_text(image, brect, handedness_label, hand_sign_text):
    cv2.rectangle(image, (brect[0], brect[1]), (brect[2], brect[1]-22), (0,0,0), -1)
    info_text = handedness_label + ':' + hand_sign_text
    cv2.putText(image, info_text, (brect[0]+5, brect[1]-4),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,255), 1, cv2.LINE_AA)
    return image

print('Helper functions ready.')

Helper functions ready.


## Step 5: Load Models and Labels

This step initializes MediaPipe, loads the trained MLP classifier, reads the gesture label names, and defines the gesture-to-note mapping used for music generation.

In [ ]:
# Download the hand landmark model bundle (MediaPipe Tasks API)
!wget -q -N -O hand_landmarker.task https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/latest/hand_landmarker.task

base_options = mp_python.BaseOptions(model_asset_path='hand_landmarker.task')
hand_landmarker_options = vision.HandLandmarkerOptions(
    base_options=base_options,
    num_hands=2,
    min_hand_detection_confidence=0.7,
    min_hand_presence_confidence=0.5,
    min_tracking_confidence=0.5,
)
hand_landmarker = vision.HandLandmarker.create_from_options(hand_landmarker_options)

keypoint_classifier = KeyPointClassifier()
point_history_classifier = PointHistoryClassifier()

with open('model/keypoint_classifier/keypoint_classifier_label.csv', encoding='utf-8-sig') as f:
    keypoint_classifier_labels = [row[0] for row in csv.reader(f)]

with open('model/point_history_classifier/point_history_classifier_label.csv', encoding='utf-8-sig') as f:
    point_history_classifier_labels = [row[0] for row in csv.reader(f)]

history_length = 16
point_history = deque(maxlen=history_length)
finger_gesture_history = deque(maxlen=history_length)

# Gesture frequencies for Web Audio
GESTURE_NOTES = {
    'Open':         'C4',
    'Close':        'C5',
    'Pointer':      'D4',
    'OK':           'G4',
    'Peace':        'E4',
    'ThreeFingers': 'F4',
    'ThumbsUp':     'A4',
    'PinkyUp':      'Bb4',
}

print('Models loaded. Gesture classes:', keypoint_classifier_labels)


## Step 5.1: Unlock Browser Audio

Browsers block audio playback unless it follows a direct user interaction (a click). Run this cell to display a button and **click it once** to unlock audio before starting the stream. Without this step, Tone.js will not play any sounds.

In [6]:
display(Javascript('''
    document.body.innerHTML += '<button onclick="var c=new AudioContext();c.resume();this.innerText=\\'Audio unlocked!\\';this.disabled=true;" style="padding:10px 20px;font-size:16px;cursor:pointer;">🔊 Click to Enable Audio</button>';
'''))

<IPython.core.display.Javascript object>

## Step 6: Live Streaming Demo

This is the main demo cell. When run it will:
1. Set up a persistent webcam stream in your browser using JavaScript
2. Load **Tone.js** (audio synthesis library) from CDN for musical note playback
3. Enter a continuous loop: capture frame → run MediaPipe → classify gesture → play note → display annotated frame

**Before running:**
- Make sure you clicked the audio unlock button in Step 5.1
- Allow camera access when your browser prompts

**To stop:** press the **■ Stop button** (interrupt kernel) in the toolbar

**Expected performance:** ~2–5 FPS in Colab due to the JS↔Python bridge latency

In [ ]:
# This JS sets up a persistent video stream in the browser.
# Each frame is captured on demand when Python calls eval_js('getFrame()').
setup_js = Javascript('''
    window._gestureStream = null;
    window._gestureVideo = null;
    window._gestureCanvas = null;
    window._gestureDisplay = null;

    async function setupStream() {
        // Create video element
        const video = document.createElement('video');
        video.setAttribute('playsinline', '');
        video.style.display = 'none';
        document.body.appendChild(video);

        // Create capture canvas (hidden)
        const captureCanvas = document.createElement('canvas');
        captureCanvas.style.display = 'none';
        document.body.appendChild(captureCanvas);

        // Create display canvas (visible)
        const displayCanvas = document.createElement('canvas');
        displayCanvas.style.border = '2px solid #333';
        displayCanvas.style.borderRadius = '8px';
        displayCanvas.style.marginTop = '8px';
        document.querySelector('#output-area') && document.querySelector('#output-area').appendChild(displayCanvas);
        document.body.appendChild(displayCanvas);

        // Start webcam
        const stream = await navigator.mediaDevices.getUserMedia({video: {width: 320, height: 240}});
        video.srcObject = stream;
        await video.play();
        await new Promise(r => setTimeout(r, 1000));

        captureCanvas.width = video.videoWidth;
        captureCanvas.height = video.videoHeight;
        displayCanvas.width = video.videoWidth;
        displayCanvas.height = video.videoHeight;

        window._gestureStream = stream;
        window._gestureVideo = video;
        window._gestureCanvas = captureCanvas;
        window._gestureDisplay = displayCanvas;

        return video.videoWidth + 'x' + video.videoHeight;
    }

    function getFrame() {
        if (!window._gestureVideo || !window._gestureCanvas) return '';
        const ctx = window._gestureCanvas.getContext('2d');
        ctx.drawImage(window._gestureVideo, 0, 0);
        return window._gestureCanvas.toDataURL('image/jpeg', 0.7);
    }

    function showFrame(dataUrl) {
        if (!window._gestureDisplay) return;
        const img = new Image();
        img.onload = () => {
            window._gestureDisplay.getContext('2d').drawImage(img, 0, 0);
        };
        img.src = dataUrl;
    }

    // function playTone(freq) {
    //     const ctx = new (window.AudioContext || window.webkitAudioContext)();
    //     const osc = ctx.createOscillator();
    //     const gain = ctx.createGain();
    //     osc.connect(gain);
    //     gain.connect(ctx.destination);
    //     osc.type = 'sine';
    //     osc.frequency.value = freq;
    //     gain.gain.setValueAtTime(0.5, ctx.currentTime);
    //     gain.gain.exponentialRampToValueAtTime(0.001, ctx.currentTime + 0.5);
    //     osc.start(ctx.currentTime);
    //     osc.stop(ctx.currentTime + 0.5);
    // }

    async function loadTone() {
        await new Promise(r => {
            const script = document.createElement('script');
            script.src = 'https://cdnjs.cloudflare.com/ajax/libs/tone/14.8.49/Tone.js';
            script.onload = r;
            document.head.appendChild(script);
        });
        window._synth = new Tone.Synth({
            oscillator: { type: 'triangle' },
            envelope: { attack: 0.02, decay: 0.1, sustain: 0.3, release: 0.5 }
        }).toDestination();
        console.log('Tone.js ready');
    }

    function playTone(note) {
        if (!window._synth) return;
        Tone.start();
        window._synth.triggerAttackRelease(note, '8n');
    }

    loadTone();

    function stopStream() {
        if (window._gestureStream) {
            window._gestureStream.getTracks().forEach(t => t.stop());
        }
    }

    setupStream();
''')

display(setup_js)
print('Setting up webcam stream... allow camera access in your browser.')
time.sleep(2)  # Wait for JS setup
print('Stream ready. Starting gesture detection loop...')
print('Interrupt the kernel (■ Stop button) to stop.')

last_gesture = None
last_gesture_time = 0
frame_count = 0

try:
    while True:
        # Capture frame from browser
        data_url = eval_js('getFrame()')
        if not data_url:
            time.sleep(0.1)
            continue

        # Decode to numpy array
        img_bytes = base64.b64decode(data_url.split(',')[1])
        img_array = np.frombuffer(img_bytes, dtype=np.uint8)
        image = cv2.imdecode(img_array, cv2.IMREAD_COLOR)
        if image is None:
            continue

        image = cv2.flip(image, 1)
        debug_image = copy.deepcopy(image)

        # Run MediaPipe (Tasks API)
        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=np.ascontiguousarray(image_rgb))
        results = hand_landmarker.detect(mp_image)

        detected_gesture = ''

        if results.hand_landmarks:
            for hand_landmarks, handedness in zip(results.hand_landmarks, results.handedness):
                handedness_label = handedness[0].category_name if handedness else '?'
                brect = calc_bounding_rect(debug_image, hand_landmarks)
                landmark_list = calc_landmark_list(debug_image, hand_landmarks)
                pre_processed = pre_process_landmark(landmark_list)
                pre_processed_ph = pre_process_point_history(debug_image, point_history)

                hand_sign_id = keypoint_classifier(pre_processed)
                hand_sign_text = keypoint_classifier_labels[hand_sign_id]
                detected_gesture = hand_sign_text

                if hand_sign_id == 2:
                    point_history.append(landmark_list[8])
                else:
                    point_history.append([0, 0])

                # Play sound when gesture changes
                import time
                current_time = time.time()
                if hand_sign_text in GESTURE_NOTES:
                    if hand_sign_text != last_gesture or (current_time - last_gesture_time) > 1.0:
                        eval_js(f'playTone("{GESTURE_NOTES[hand_sign_text]}")')
                        last_gesture = hand_sign_text
                        last_gesture_time = current_time

                debug_image = draw_bounding_rect(debug_image, brect)
                debug_image = draw_landmarks(debug_image, landmark_list)
                debug_image = draw_info_text(debug_image, brect, handedness_label, hand_sign_text)
        else:
            point_history.append([0, 0])
            last_gesture = None

        # Overlay gesture label
        label = detected_gesture if detected_gesture else 'No hand detected'
        cv2.putText(debug_image, f'Gesture: {label}', (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0,0,0), 4, cv2.LINE_AA)
        cv2.putText(debug_image, f'Gesture: {label}', (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0,255,0), 2, cv2.LINE_AA)

        # Encode annotated frame back to base64 and send to browser canvas
        _, buffer = cv2.imencode('.jpg', debug_image, [cv2.IMWRITE_JPEG_QUALITY, 50])
        encoded = base64.b64encode(buffer).decode('utf-8')
        display(Javascript(f'showFrame("data:image/jpeg;base64,{encoded}")'))

        frame_count += 1

except KeyboardInterrupt:
    display(Javascript('stopStream()'))
    print(f'\nStream stopped. Processed {frame_count} frames.')